# Extracting Frames from Deep Sea Video

This notebook walks through how to use `soi-frame-extractor` to pull still images from video files, with sensor data (position, depth, temperature, salinity, etc.) embedded directly into each image.

**What you need:**
- One or more video files (`.mp4` or `.mov`) in a folder
- Optionally: a CSV file from your vehicle's data system with timestamps and sensor readings
- An extraction spec: a small text file describing how often to extract frames and any conditions to apply

**What you get:**
- JPEG images in an output folder, named by timestamp
- Sensor metadata embedded into each image (GPS coordinates, depth, etc.)
- An `ifdo.json` sidecar describing the full dataset
- A `biigle_metadata.csv` ready to upload to BIIGLE

## Installation

Run this once in your terminal before opening this notebook:

```
uv pip install git+https://github.com/schmidtocean/soi-frame-extractor
```

If `uv` is not installed, see https://docs.astral.sh/uv/getting-started/installation/ or install using pip:

```
git clone https://github.com/schmidtocean/soi-frame-extractor
cd soi-frame-extractor
pip install -e .
```

## Setup

Edit the code block below to set directories and spec location:

In [ ]:
from pathlib import Path

# Point these at your actual files and folders.
# Path("video/") means a folder called "video" in the same directory as this notebook.
# You can also use an absolute path, e.g. Path("/home/thomas/data/dive42/video/")

video_dir  = Path("video/")      # folder containing your .mp4 or .mov files
spec_file  = Path("spec.yaml")   # your extraction spec (see below)
output_dir = Path("frames/")     # where extracted frames will be written

# Create the output folder if it doesn't exist yet
output_dir.mkdir(exist_ok=True)

## The Extraction Spec

The extraction spec is a plain text file (YAML format) that tells the tool how to extract frames.
Save it as `spec.yaml` in the same folder as this notebook, or update the path in the cell above.

A minimal spec — one frame every 10 seconds:

```yaml
rules:
  - interval_s: 10.0
```

A fuller example with sensor data and project metadata:

```yaml
rules:
  - interval_s: 10.0

mappings:
  timestamp:  Timestamp        # the column in your CSV that holds the time
  latitude:   Latitude_ddeg    # adjust these to match your CSV column names
  longitude:  Longitude_ddeg
  depth:      Depth_m

metadata:
  cruise_id: FK250101
  dive_id:   S0042
  vehicle:   SuBastian
```

The `metadata` block is optional — any values you include will be embedded into every extracted frame.

A complete example of all available spec options is available in the repository as `extraction_spec.yaml`.
Note that the CSV mapping section writes specific fields (latitude, longitude, depth, etc.) to the correct
fields in EXIF, iFDO, and BIIGLE CSV automatically. See `extraction_spec.yaml` for more details.

## Example 1: Extract frames at a fixed interval

The simplest case — one frame every 10 seconds, no sensor data required.
Point `video_dir` at your folder of videos and make sure `spec.yaml` exists, then run the cell below.

In [ ]:
from soi_frame_extractor.pipeline import extract

extract(
    spec_path=spec_file,
    video_source=video_dir,
    output_dir=output_dir,
)

frames = sorted(output_dir.glob("*.jpg"))
print(f"Done. {len(frames)} frames written to {output_dir}/")

## What's in the output folder?

After running you will find:

| File | What it is |
|---|---|
| `*.jpg` | Extracted frames, one per planned timestamp |
| `ifdo.json` | Dataset manifest — one entry per frame with all metadata |
| `biigle_metadata.csv` | Ready to upload directly to BIIGLE |

## Example 2: With sensor data

If you have a CSV from your vehicle's data system, pass it to `extract()` with `csv_path`.
The tool will interpolate sensor values at each frame's exact timestamp and embed them in the image.

Make sure your `spec.yaml` includes a `mappings` block that matches your CSV column names (see above).

In [ ]:
csv_file = Path("data/sensors.csv")   # update this to your sensor CSV

extract(
    spec_path=spec_file,
    video_source=video_dir,
    output_dir=output_dir,
    csv_path=csv_file,
)

frames = sorted(output_dir.glob("*.jpg"))
print(f"Done. {len(frames)} frames written to {output_dir}/")

Sensor values (depth, position, etc.) are embedded inside each JPEG, they travel with the image
into any downstream tool that reads EXIF or XMP. They are also written to the biggle_metadata.csv
file in the frames directory. To confirm, you can examine the BIIGLE file or open an image with
a tool that reads EXIF/XMP data.